## 1. Imports and path

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from pathlib import Path
from itertools import product           #used to produce the combinations
from scipy.stats import wilcoxon, binomtest

warnings.filterwarnings('ignore')
np.random.seed(36)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.5f}'.format)

PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models'
PHASE7_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase7_stats'
PHASE7_OUTPUT.mkdir(parents=True, exist_ok=True)

BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'
BASELINE_KEY=f'{BASELINE_MODEL}_{BASELINE_PIPELINE}'

MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']
DATASETS =['2007', '2008']

COMPARE_CONFIGS=[
    f'{m}_{p}'
    for m, p in product(MODELS, PIPELINES)      #generates all combinations
    if f'{m}_{p}' != BASELINE_KEY
]

FDR_ALPHA=0.05                  #False Discovery Rate
N_BOOTSTRAP=2000                #how many times we "resample" queries to estimate uncertainty
EFFECT_SMALL=0.10               #minimum practical effect size threshold for NDCG@5 and P@5
FAILURE_EFFECT_SMALL=0.01       #don't claim a meaningful change unless the absolute change is at least 1%

print('='*80)
print('Phase 7 = STATISTICAL VALIDATION ONLY')
print('='*80)
print(f'Phase 7 output  : {PHASE7_OUTPUT}')
print(f'Baseline config : {BASELINE_KEY}')
print(f'Configs to test : {len(COMPARE_CONFIGS)}')
print(f'Datasets        : {DATASETS}')
print(f'FDR alpha       : {FDR_ALPHA}')
print(f'Bootstrap reps  : {N_BOOTSTRAP}')
print(f'Cliff delta thresh  : {EFFECT_SMALL}')
print(f'Risk diff thresh: {FAILURE_EFFECT_SMALL}')
print('='*80)

Phase 7 = STATISTICAL VALIDATION ONLY
Phase 7 output  : b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase7_stats
Baseline config : pointwise_raw
Configs to test : 8
Datasets        : ['2007', '2008']
FDR alpha       : 0.05
Bootstrap reps  : 2000
Cliff delta thresh  : 0.1
Risk diff thresh: 0.01


## 2. Utility functions

In [ ]:
# 1. failure-flag parser 

def make_fail_flag(series: pd.Series)->pd.Series:
    #Converting Failure@5_primary to clean 0/1 int. No silent misclassification.
    if pd.api.types.is_bool_dtype(series):
        return series.astype(int)
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).astype(float).round().astype(int).clip(0, 1)
    _T = {'true', '1', 'yes', '1.0'}
    _F = {'false', '0', 'no', '0.0', ''}
    def _p(v):
        if pd.isna(v):
            return 0
        if isinstance(v, bool):
            return int(v)
        if isinstance(v, (int, float)):
            return 0 if np.isnan(float(v)) else int(round(float(v)))
        if isinstance(v, str):
            vl = v.strip().lower()
            if vl in _T: return 1
            if vl in _F: return 0
        return 0
    return series.map(_p).astype(int)

#self-test
_ts = pd.Series([True, False, 'True', 'false', '1', '0', 1.0, 0.0])
assert list(make_fail_flag(_ts))==[1,0,1,0,1,0,1,0]
del _ts
print('make_fail_flag')


# 2.  Paired merge by qid intersection 
def paired_merge(df_a: pd.DataFrame, df_b: pd.DataFrame,
                 metric: str, evaluable_only: bool = True):
    """
    Return (vec_a, vec_b, qids) aligned by qid intersection.
    If evaluable_only, restrict to qids with num_relevant_1 > 0 in BOTH.
    Raises RuntimeError if required columns are missing.
    """
    required=['qid', metric]
    if evaluable_only:
        required.append('num_relevant_1')
    for col in required:
        if col not in df_a.columns:
            raise RuntimeError(f'Missing column "{col}" in df_a')
        if col not in df_b.columns:
            raise RuntimeError(f'Missing column "{col}" in df_b')

    if evaluable_only:
        a=df_a[df_a['num_relevant_1'] > 0][['qid', metric]].copy()
        b=df_b[df_b['num_relevant_1'] > 0][['qid', metric]].copy()
    else:
        a=df_a[['qid', metric]].copy()
        b=df_b[['qid', metric]].copy()
    merged=a.merge(b, on='qid', suffixes=('_a', '_b'))
    va=merged[f'{metric}_a'].values
    vb=merged[f'{metric}_b'].values
    assert len(va)==len(vb), 'Paired vectors length mismatch after merge'
    return va, vb, merged['qid'].values

print('paired_merge')



# 3.  Wilcoxon wrapper 

def wilcoxon_test(va: np.ndarray, vb: np.ndarray):
    """
    Wilcoxon signed-rank test on paired samples (vb = config, va = baseline).
    Returns (stat, pval, note).
    Guards:
    - If all differences are zero -> p=1.0
    - If zero differences remain after filtering -> p=1.0
    """
    diff=(vb-va).astype(float)

    if np.all(diff==0):
        return 0.0, 1.0, 'no differences observed'

    #filtering zeros, then testing on nonzero
    nonzero=diff[diff != 0]
    if len(nonzero)==0:
        return 0.0, 1.0, 'all differences became zero after filtering'

    try:
        #running wilcoxon on nonzero
        stat, pval=wilcoxon(nonzero, zero_method='wilcox', alternative='two-sided')
        return float(stat), float(pval), ''
    except Exception as e:
        return 0.0, 1.0, f'wilcoxon error: {e}'

print('wilcoxon_test')



# 4.  McNemar's test (exact binomial)

def mcnemar_test(fa: np.ndarray, fb: np.ndarray):
    """
    Exact binomial McNemar on paired binary vectors.
    fa=baseline flags, fb=config flags (both 0/1 int arrays).
    Returns (b01, b10, pval, note).
    b01=baseline=0 config=1  (new failures)
    b10=baseline=1 config=0  (resolved failures)
    """
    b01=int(((fa==0) & (fb==1)).sum())
    b10=int(((fa==1) & (fb==0)).sum())
    discordant=b01+b10
    if discordant==0:
        return b01, b10, 1.0, 'no discordant pairs'
    result=binomtest(min(b01, b10), discordant, 0.5, alternative='two-sided')
    return b01, b10, float(result.pvalue), ''

print('mcnemar_test')



# 5.  Cliff's delta (from paired differences)

def cliffs_delta_paired(va: np.ndarray, vb: np.ndarray) -> float:
    """
    Cliff's delta estimated from paired differences d=vb-va.
    =(number of d>0 - number of d<0) / n
    Range: [-1, 1].  Positive -> config tends to be higher.
    """
    d=vb-va
    n=len(d)
    if n==0:
        return 0.0
    return float((np.sum(d > 0)-np.sum(d < 0)) / n)

print('cliffs_delta_paired')



# 6.  Bootstrap CI for median difference 

def bootstrap_ci_median_diff(va: np.ndarray, vb: np.ndarray,
                              n_boot: int=N_BOOTSTRAP,
                              ci: float=0.95) -> tuple:
    """
    95% bootstrap CI for median(vb-va) using percentile method.
    Returns (lower, upper).
    """
    diff=(vb - va).astype(float)
    n=len(diff)
    if n==0:
        return 0.0, 0.0
    medians=np.array([
        np.median(diff[np.random.randint(0, n, n)])
        for _ in range(n_boot)
    ], dtype=float)
    alpha=(1 - ci) / 2
    return (
        float(np.percentile(medians, 100 * alpha)),
        float(np.percentile(medians, 100 * (1 - alpha)))
    )

print('bootstrap_ci_median_diff')



# 7.  Bootstrap CI for risk difference 

def bootstrap_ci_risk_diff(fa: np.ndarray, fb: np.ndarray,
                            n_boot: int = N_BOOTSTRAP,
                            ci: float = 0.95) -> tuple:
    """
    95% bootstrap CI for risk_diff=mean(fb - fa).
    Proper paired resampling.
    """
    diff=fb-fa
    n=len(diff)
    if n==0:
        return 0.0, 0.0
    boot_stats=[]
    for _ in range(n_boot):
        idx=np.random.randint(0, n, n)
        boot_stats.append(diff[idx].mean())
    alpha=(1-ci)/2
    return (
        float(np.percentile(boot_stats, 100 * alpha)),
        float(np.percentile(boot_stats, 100 * (1 - alpha)))
    )

print('bootstrap_ci_risk_diff')



# 8.  Benjamini-Hochberg FDR correction 

def bh_fdr(pvals: list, alpha: float = FDR_ALPHA) -> np.ndarray:
    """
    Benjamini-Hochberg FDR 
    Returns array of adjusted q-values (same order as input).
    """
    pvals=np.asarray(pvals, dtype=float)
    n=len(pvals)
    if n==0:
        return np.array([])
    order=np.argsort(pvals)
    ranks=np.arange(1, n + 1)
    adj=np.minimum(pvals[order] * n / ranks, 1.0)
    adj=np.minimum.accumulate(adj[::-1])[::-1]
    q=np.empty(n)
    q[order]=adj
    return q

#quick test
_pv=[0.001, 0.002, 0.5, 0.04, 0.8]
_q=bh_fdr(_pv)
assert _q[0]<_q[2], 'BH ordering broken'
del _pv, _q
print('bh_fdr')

print('\nAll utility functions defined')

make_fail_flag
paired_merge
wilcoxon_test
mcnemar_test
cliffs_delta_paired
bootstrap_ci_median_diff
bootstrap_ci_risk_diff
bh_fdr

All utility functions defined


Wilcoxon -> Is improvement statistically detectable?

Cliff’s delta -> Is improvement strong and consistent?

Bootstrap CI -> Is improvement stable?

Risk difference threshold -> Is improvement practically meaningful?

## 3. Loading Phase 6 artifacts

In [7]:
print('='*80)
print('LOADING PHASE 6 QUERY METRICS')
print('='*80)

qm:dict[str, pd.DataFrame]={}

for model, pipeline, dataset in product(MODELS, PIPELINES, DATASETS):
    key =f'{model}_{pipeline}_{dataset}'
    fpath=PHASE6_OUTPUT / f'{key}_query_metrics.csv'
    if not fpath.exists():
        raise RuntimeError(f'Missing required file: {fpath}')
    df=pd.read_csv(fpath)
    df['_fail_flag']=make_fail_flag(df['Failure@5_primary'])
    qm[key]=df
    print(f'{key:40s}  ({len(df)} queries)')

expected=len(MODELS)*len(PIPELINES)*len(DATASETS)
if len(qm)!=expected:
    raise RuntimeError(f'Expected {expected} files, loaded {len(qm)}')

print(f'\nLoaded {len(qm)} / {expected} files')
print('='*80)

LOADING PHASE 6 QUERY METRICS
pointwise_raw_2007                        (336 queries)
pointwise_raw_2008                        (156 queries)
pointwise_global_2007                     (336 queries)
pointwise_global_2008                     (156 queries)
pointwise_per_query_2007                  (336 queries)
pointwise_per_query_2008                  (156 queries)
pairwise_raw_2007                         (336 queries)
pairwise_raw_2008                         (156 queries)
pairwise_global_2007                      (336 queries)
pairwise_global_2008                      (156 queries)
pairwise_per_query_2007                   (336 queries)
pairwise_per_query_2008                   (156 queries)
lightgbm_raw_2007                         (336 queries)
lightgbm_raw_2008                         (156 queries)
lightgbm_global_2007                      (336 queries)
lightgbm_global_2008                      (156 queries)
lightgbm_per_query_2007                   (336 queries)
lightgbm_per_query

## 4. Running NDCG@5 and P@5 tests (Wilcoxon + Effect size + CI)

In [10]:
print('='*80)
print('NDCG@5 AND P@5_PRIMARY: WILCOXON PAIRED TESTS VS BASELINE')
print('='*80)

ndcg_rows=[]
p5_rows=[]
warnings_log=[]

for dataset in DATASETS:
    baseline_key=f'{BASELINE_KEY}_{dataset}'
    df_base=qm[baseline_key]

    for cfg in COMPARE_CONFIGS:
        cfg_key=f'{cfg}_{dataset}'
        df_cfg =qm[cfg_key]
        model, pipeline=cfg.split('_', 1)

        for metric, rows_list in [('NDCG@5', ndcg_rows), ('P@5_primary', p5_rows)]:
            va, vb, qids=paired_merge(df_base, df_cfg, metric, evaluable_only=True)
            n=len(va)
            diff=vb-va
            med_diff=float(np.median(diff))

            
            #Always call wilcoxon_test (it handles zero-diff internally)
            stat, pval, note=wilcoxon_test(va, vb)

            cd=cliffs_delta_paired(va, vb)
            ci_lo, ci_hi=bootstrap_ci_median_diff(va, vb)

            if note:
                warnings_log.append(f'{cfg_key} {metric}: {note}')

            rows_list.append({
                'dataset':dataset,
                'model':model,
                'pipeline':pipeline,
                'config':cfg,
                'metric':metric,
                'n_queries':n,
                'median_diff':med_diff,
                'ci_lo_95':ci_lo,
                'ci_hi_95':ci_hi,
                'cliffs_delta':cd,
                'wilcoxon_stat':stat,
                'pval_raw':pval,
                'note':note
            })

ndcg_df=pd.DataFrame(ndcg_rows)
p5_df=pd.DataFrame(p5_rows)

#Applying BH-FDR correction per (dataset, metric)
for df_res in [ndcg_df, p5_df]:
    df_res['qval_fdr']=np.nan
    for dataset in DATASETS:
        mask=df_res['dataset']==dataset
        df_res.loc[mask, 'qval_fdr']=bh_fdr(df_res.loc[mask, 'pval_raw'].values)

#Adding direction column
ndcg_df['direction']=np.sign(ndcg_df['median_diff'])
p5_df['direction']=np.sign(p5_df['median_diff'])

print('\nNDCG@5 results (MQ2007):')
display(ndcg_df[ndcg_df['dataset']=='2007']
        [['model','pipeline','n_queries','median_diff','ci_lo_95','ci_hi_95',
          'cliffs_delta','pval_raw','qval_fdr','direction','note']]
        .sort_values(['model','pipeline']))

print('\nNDCG@5 results (MQ2008):')
display(ndcg_df[ndcg_df['dataset']=='2008']
        [['model','pipeline','n_queries','median_diff','ci_lo_95','ci_hi_95',
          'cliffs_delta','pval_raw','qval_fdr','direction','note']]
        .sort_values(['model','pipeline']))

print('\nP@5_primary results (MQ2007):')
display(p5_df[p5_df['dataset']=='2007']
        [['model','pipeline','n_queries','median_diff','ci_lo_95','ci_hi_95',
          'cliffs_delta','pval_raw','qval_fdr','direction','note']]
        .sort_values(['model','pipeline']))

print('\nP@5_primary results (MQ2008):')
display(p5_df[p5_df['dataset']=='2008']
        [['model','pipeline','n_queries','median_diff','ci_lo_95','ci_hi_95',
          'cliffs_delta','pval_raw','qval_fdr','direction','note']]
        .sort_values(['model','pipeline']))


print(f'\nNDCG rows: {len(ndcg_df)}  |  P@5 rows: {len(p5_df)}')

NDCG@5 AND P@5_PRIMARY: WILCOXON PAIRED TESTS VS BASELINE

NDCG@5 results (MQ2007):


,model,pipeline,n_queries,median_diff,ci_lo_95,ci_hi_95,cliffs_delta,pval_raw,qval_fdr,direction,note
6,lightgbm,global,290,0.00000,0.00000,0.00000,-0.00690,0.81373,0.86574,0.00000,
7,lightgbm,per_query,290,0.00000,0.00000,0.00000,0.07931,0.10353,0.15975,0.00000,
5,lightgbm,raw,290,0.00000,0.00000,0.00000,-0.00345,0.86574,0.86574,0.00000,
3,pairwise,global,290,0.00000,0.00000,0.00000,0.09310,0.00793,0.01586,0.00000,
4,pairwise,per_query,290,0.00000,0.00000,0.00000,0.12759,0.00548,0.01462,0.00000,
2,pairwise,raw,290,0.00000,0.00000,0.00000,0.10345,0.00491,0.01462,0.00000,
0,pointwise,global,290,0.00000,0.00000,0.00000,-0.02414,0.11981,0.15975,0.00000,
1,pointwise,per_query,290,0.00000,0.00000,0.00000,0.11379,0.00412,0.01462,0.00000,



NDCG@5 results (MQ2008):


,model,pipeline,n_queries,median_diff,ci_lo_95,ci_hi_95,cliffs_delta,pval_raw,qval_fdr,direction,note
14,lightgbm,global,105,0.00000,-0.01486,0.00000,-0.03810,0.80111,0.91556,0.00000,
15,lightgbm,per_query,105,0.00000,0.00000,0.00000,-0.02857,0.94527,0.94527,0.00000,
13,lightgbm,raw,105,0.00000,0.00000,0.01711,0.07619,0.62084,0.82779,0.00000,
11,pairwise,global,105,-0.13121,-0.22187,-0.04857,-0.39048,0.00000,0.00000,-1.00000,
12,pairwise,per_query,105,0.00000,-0.04374,0.00000,-0.15238,0.07497,0.11995,0.00000,
10,pairwise,raw,105,0.00000,-0.02258,0.00000,-0.18095,0.01034,0.02069,0.00000,
8,pointwise,global,105,-0.02581,-0.15633,0.00000,-0.31429,0.00001,0.00004,-1.00000,
9,pointwise,per_query,105,0.00000,-0.05111,0.00000,-0.20000,0.00780,0.02069,0.00000,



P@5_primary results (MQ2007):


,model,pipeline,n_queries,median_diff,ci_lo_95,ci_hi_95,cliffs_delta,pval_raw,qval_fdr,direction,note
6,lightgbm,global,290,0.00000,0.00000,0.00000,-0.02759,0.82351,0.83366,0.00000,
7,lightgbm,per_query,290,0.00000,0.00000,0.00000,0.05517,0.19189,0.28114,0.00000,
5,lightgbm,raw,290,0.00000,0.00000,0.00000,0.02069,0.83366,0.83366,0.00000,
3,pairwise,global,290,0.00000,0.00000,0.00000,0.09310,0.01410,0.05641,0.00000,
4,pairwise,per_query,290,0.00000,0.00000,0.00000,0.08276,0.00827,0.05641,0.00000,
2,pairwise,raw,290,0.00000,0.00000,0.00000,0.09310,0.02223,0.05928,0.00000,
0,pointwise,global,290,0.00000,0.00000,0.00000,-0.03103,0.21086,0.28114,0.00000,
1,pointwise,per_query,290,0.00000,0.00000,0.00000,0.05862,0.04507,0.09014,0.00000,



P@5_primary results (MQ2008):


,model,pipeline,n_queries,median_diff,ci_lo_95,ci_hi_95,cliffs_delta,pval_raw,qval_fdr,direction,note
14,lightgbm,global,105,0.00000,0.00000,0.00000,-0.04762,0.16837,0.33675,0.00000,
15,lightgbm,per_query,105,0.00000,0.00000,0.00000,-0.04762,0.70133,0.73373,0.00000,
13,lightgbm,raw,105,0.00000,0.00000,0.00000,0.00000,0.48985,0.65313,0.00000,
11,pairwise,global,105,0.00000,-0.20000,0.00000,-0.40000,0.00000,0.00000,0.00000,
12,pairwise,per_query,105,0.00000,0.00000,0.00000,-0.04762,0.32046,0.51273,0.00000,
10,pairwise,raw,105,0.00000,0.00000,0.00000,-0.03810,0.73373,0.73373,0.00000,
8,pointwise,global,105,0.00000,-0.20000,0.00000,-0.36190,0.00000,0.00000,0.00000,
9,pointwise,per_query,105,0.00000,0.00000,0.00000,-0.20000,0.00010,0.00026,0.00000,



NDCG rows: 16  |  P@5 rows: 16


## Interpretation


We are comparing each Phase 6 system against our baseline 

For each config, we ask:
1) **Is the difference real (not just random luck)?**  
   -> Check **qval_fdr < 0.05** (this is the “significant after correction” rule)
2) **Is the difference big enough to matter?**  
   -> Check **|Cliff’s delta| >= 0.10** (this is the “meaningful effect” rule)

We only claim a meaningful result if **both** are true.



- **Significant (q < 0.05)** means:  
  "Across queries, this config behaves differently than the baseline in a consistent way."
- **Not significant** means:  
  "We do not have strong evidence that it is different from baseline (could be noise)."

NOTE: Significant does **not** automatically mean “better.”  
It can also mean “consistently worse.”


## Results summary for NDCG@5

### MQ2007 (dataset = 2007)
Some configs are **statistically significant** and also pass the **effect-size rule**, meaning they show a small but consistent difference vs baseline:

**Meaningfully different (passes both rules):**
- **pairwise_raw** (q=0.0146, delta=+0.103) -> small but consistent improvement tendency
- **pairwise_per_query** (q=0.0146, delta=+0.128) -> small but consistent improvement tendency
- **pointwise_per_query** (q=0.0146, delta=+0.114) -> small but consistent improvement tendency

**Significant but NOT meaningful (fails effect threshold):**
- **pairwise_global** (q=0.0159 but delta=+0.093 < 0.10) -> too small to claim

**No clear evidence of difference:**
- All **LightGBM** variants (q high, delta small)
- **pointwise_global** (not significant)

What this means for the project (MQ2007, NDCG@5):
- A few configs show a **small, consistent improvement pattern** over baseline.
- But the “typical query” did not noticeably improve 



## MQ2008 (dataset = 2008)
Here, the strongest signal is **harm**, not improvement:

**Meaningfully WORSE than baseline (passes both rules, negative effect):**
- **pairwise_global** (median_diff = -0.131, CI fully negative, delta= -0.390, q=0.0000) -> clearly worse
- **pointwise_global** (median_diff = -0.026, delta= -0.314, q=0.00004) -> worse
- **pairwise_raw** (delta= -0.181, q=0.0207) -> worse tendency
- **pointwise_per_query** (delta= -0.200, q=0.0207) -> worse tendency

**No clear evidence of difference:**
- LightGBM variants (q high, delta small)

What this means for the project (MQ2008, NDCG@5):
- Some configs that looked "okay" on MQ2007 become **meaningfully worse** on MQ2008.
- This is evidence of **generalization risk** (we should not claim “overall improvement” from Phase 6).






## Results summary for P@5_primary

### MQ2007 (dataset = 2007)
**No config is meaningfully different from baseline** on P@5_primary.
- Because for every config, **qval_fdr is NOT < 0.05** (most are around 0.056, 0.059, 0.090, etc.)
- Even when p-values look small, after FDR correction they are not below 0.05.
- Also Cliff’s delta values are mostly below 0.10.

What this means for the project:
- On MQ2007, none of the stronger models clearly improved the "top-5 precision" compared to baseline.
- So we cannot claim that Phase 6 systems improve P@5_primary on MQ2007.



### MQ2008 (dataset = 2008)
Some configs are **meaningfully worse** than baseline on P@5_primary:

**Meaningfully worse (passes both rules):**
- **pairwise_global**: qval_fdr = 0.00000 and Cliff’s delta= -0.400  
- **pointwise_global**: qval_fdr = 0.00000 and Cliff’s delta= -0.362  
- **pointwise_per_query**: qval_fdr = 0.00026 and Cliff’s delta= -0.200  

Negative Cliff’s delta means:  
“More queries got worse than got better.”

What this means for the project:
- On MQ2008 (generalization), some normalization-based configs **hurt** top-5 precision compared to baseline.
- This is important evidence that some “stronger setups” do not generalize well.

**Not meaningfully different (not significant and/or too small):**
- LightGBM variants: not significant, small effect sizes
- pairwise_raw and pairwise_per_query: not significant, small effect sizes



## Why is median_diff = 0 for almost everything? 
 especially for **P@5_primary**.

Reason: P@5 can only take a few discrete values:
- 0.0, 0.2, 0.4, 0.6, 0.8, 1.0

So the difference (config - baseline) is often:
- 0.0 (no change), or +-0.2, +-0.4, ...

If most queries don’t change at all, the **median** difference becomes **0** and stays **0** even in bootstrap.

So:
- median_diff = 0 does **not** mean “nothing changed for all queries”
- it means “at least half the queries had no change”

That’s normal for discrete metrics like P@5.




Phase 6 shows **small, limited improvements on MQ2007 for NDCG@5**, but these do **not consistently generalize**, and several configs are **meaningfully worse on MQ2008**, so we must report improvements cautiously and emphasize generalization risk.

For **P@5_primary**, Phase 6 does **not** show meaningful improvement over baseline on MQ2007, and on MQ2008 some configs are actually **significantly and meaningfully worse**, which matters because it shows generalization risk.


**"Some Phase 6 configurations show statistically detectable changes vs baseline, but improvements are not consistent across MQ2007 and MQ2008."**

**"Global normalization (and especially pairwise_global) is associated with significant degradation in MQ2008 on both NDCG@5 and P@5_primary."**

**"For P@5_primary, we do not observe meaningful improvements over baseline; generalization often worsens."**

## 5. Running Failure@5 tests (McNemar + Risk difference + CI)

In [12]:
print('='*80)
print('FAILURE@5_PRIMARY: McNEMAR PAIRED BINARY TESTS VS BASELINE')
print('='*80)

fail_rows=[]

for dataset in DATASETS:
    baseline_key=f'{BASELINE_KEY}_{dataset}'
    df_base=qm[baseline_key]

    for cfg in COMPARE_CONFIGS:
        cfg_key=f'{cfg}_{dataset}'
        df_cfg=qm[cfg_key]
        model, pipeline=cfg.split('_', 1)

        fa_raw, fb_raw, _=paired_merge(df_base, df_cfg, '_fail_flag',
                                         evaluable_only=True)
        n=len(fa_raw)
        fa=fa_raw.astype(int)
        fb=fb_raw.astype(int)

        b01, b10, pval, note=mcnemar_test(fa, fb)
        risk_diff=float(fb.mean()-fa.mean())
        ci_lo, ci_hi=bootstrap_ci_risk_diff(fa, fb)

        if note:
            warnings_log.append(f'{cfg_key} failure: {note}')

        fail_rows.append({
            'dataset':dataset,
            'model':model,
            'pipeline':pipeline,
            'config':cfg,
            'n_queries':n,
            'baseline_fail_rate':float(fa.mean()),
            'config_fail_rate':float(fb.mean()),
            'risk_diff':risk_diff,
            'ci_lo_95':ci_lo,
            'ci_hi_95':ci_hi,
            'b01_new_fails':b01,
            'b10_resolved':b10,
            'pval_raw':pval,
            'note':note
        })

fail_df=pd.DataFrame(fail_rows)

#BH-FDR per dataset
fail_df['qval_fdr']=np.nan
for dataset in DATASETS:
    mask=fail_df['dataset']==dataset
    fail_df.loc[mask, 'qval_fdr']=bh_fdr(fail_df.loc[mask, 'pval_raw'].values)

#Adding direction column
fail_df['direction']=np.sign(fail_df['risk_diff'])

print('\nFailure@5 results (MQ2007):')
display(fail_df[fail_df['dataset']=='2007']
        [['model','pipeline','n_queries','baseline_fail_rate','config_fail_rate',
          'risk_diff','ci_lo_95','ci_hi_95','b01_new_fails','b10_resolved',
          'pval_raw','qval_fdr','direction','note']]
        .sort_values(['model','pipeline']))

print('\nFailure@5 results (MQ2008):')
display(fail_df[fail_df['dataset']=='2008']
        [['model','pipeline','n_queries','baseline_fail_rate','config_fail_rate',
          'risk_diff','ci_lo_95','ci_hi_95','b01_new_fails','b10_resolved',
          'pval_raw','qval_fdr','direction','note']]
        .sort_values(['model','pipeline']))

print(f'\nFailure rows: {len(fail_df)}')

FAILURE@5_PRIMARY: McNEMAR PAIRED BINARY TESTS VS BASELINE

Failure@5 results (MQ2007):


,model,pipeline,n_queries,baseline_fail_rate,config_fail_rate,risk_diff,ci_lo_95,ci_hi_95,b01_new_fails,b10_resolved,pval_raw,qval_fdr,direction,note
6,lightgbm,global,290,0.15172,0.18276,0.03103,-0.01034,0.06905,22,13,0.17547,0.43592,1.00000,
7,lightgbm,per_query,290,0.15172,0.15172,0.00000,-0.04138,0.03793,16,16,1.00000,1.00000,0.00000,
5,lightgbm,raw,290,0.15172,0.17586,0.02414,-0.01379,0.06207,20,13,0.29621,0.43592,1.00000,
3,pairwise,global,290,0.15172,0.17241,0.02069,-0.01379,0.05862,16,10,0.32694,0.43592,1.00000,
4,pairwise,per_query,290,0.15172,0.17586,0.02414,-0.01379,0.06552,19,12,0.28104,0.43592,1.00000,
2,pairwise,raw,290,0.15172,0.17241,0.02069,-0.01034,0.05517,16,10,0.32694,0.43592,1.00000,
0,pointwise,global,290,0.15172,0.15862,0.00690,0.00000,0.01724,2,0,0.50000,0.57143,1.00000,
1,pointwise,per_query,290,0.15172,0.19310,0.04138,0.00690,0.07586,20,8,0.03570,0.28559,1.00000,



Failure@5 results (MQ2008):


,model,pipeline,n_queries,baseline_fail_rate,config_fail_rate,risk_diff,ci_lo_95,ci_hi_95,b01_new_fails,b10_resolved,pval_raw,qval_fdr,direction,note
14,lightgbm,global,105,0.07619,0.10476,0.02857,-0.01905,0.07619,5,2,0.45312,0.60417,1.00000,
15,lightgbm,per_query,105,0.07619,0.11429,0.03810,-0.00952,0.08571,6,2,0.28906,0.46250,1.00000,
13,lightgbm,raw,105,0.07619,0.09524,0.01905,-0.02857,0.06667,4,2,0.68750,0.78571,1.00000,
11,pairwise,global,105,0.07619,0.19048,0.11429,0.03810,0.19048,15,3,0.00754,0.06030,1.00000,
12,pairwise,per_query,105,0.07619,0.08571,0.00952,-0.01905,0.03810,2,1,1.00000,1.00000,1.00000,
10,pairwise,raw,105,0.07619,0.10476,0.02857,0.00000,0.06667,3,0,0.25000,0.46250,1.00000,
8,pointwise,global,105,0.07619,0.15238,0.07619,0.01905,0.14286,10,2,0.03857,0.15430,1.00000,
9,pointwise,per_query,105,0.07619,0.12381,0.04762,-0.00952,0.10476,7,2,0.17969,0.46250,1.00000,



Failure rows: 16


**Decision rule :** We only claim a meaningful change if BOTH are true:
- **qval_fdr < 0.05** (significant after BH-FDR within the dataset)
- **|risk_diff| >= 0.01** (practically meaningful change)

Where:
- **risk_diff = config_fail_rate − baseline_fail_rate**
  - **risk_diff < 0** -> fewer failures (good)
  - **risk_diff > 0** -> more failures (bad)


## MQ2007 (n=290 evaluable queries)
- Baseline failure rate is **0.15172 (15.2%)**.
- All configs have **qval_fdr ≥ 0.28** (most ≈ 0.44 or 1.00), so **none are statistically significant after correction**.
- Many configs show **positive risk_diff** (higher failure rate than baseline), but the bootstrap CIs often include 0, and the McNemar tests are not significant after FDR.

**Conclusion (MQ2007):**  
**No Phase 6 configuration shows a statistically validated change (improvement or harm) in Failure@5_primary vs baseline.**


## MQ2008 (n=105 evaluable queries)
- Baseline failure rate is **0.07619 (7.6%)**.
- All configs have **qval_fdr ≥ 0.06**, so **none are statistically significant after correction**.
- One config (**pairwise_global**) looks concerning (failure rate rises to **0.19048**, risk_diff **+0.11429**, many more new failures than resolved), but its **qval_fdr = 0.06030**, which fails the strict cutoff.

**Conclusion (MQ2008):**  
**No Phase 6 configuration shows a statistically validated change (improvement or harm) in Failure@5_primary vs baseline.**  
Some patterns suggest possible harm, but they are **not confirmed** under the FDR rule.



## Overall conclusion for Failure@5_primary (Phase 7)
Across both datasets:
- **No configuration achieves a statistically significant AND practically meaningful reduction in failure rate.**
- Therefore, **we cannot claim Phase 6 reduced Failure@5_primary** relative to the baseline.


## 6. Saving Artifacts

In [14]:
print('='*80)
print('SAVING CSV ARTIFACTS')
print('='*80)

#Deterministic sort before saving
sort_cols=['dataset', 'model', 'pipeline']

ndcg_df.sort_values(sort_cols).to_csv(
    PHASE7_OUTPUT / 'phase7_stats_ndcg_vs_baseline.csv', index=False
)
print('Saved: phase7_stats_ndcg_vs_baseline.csv')

fail_df.sort_values(sort_cols).to_csv(
    PHASE7_OUTPUT / 'phase7_stats_failure_vs_baseline.csv', index=False
)
print('Saved: phase7_stats_failure_vs_baseline.csv')

p5_df.sort_values(sort_cols).to_csv(
    PHASE7_OUTPUT / 'phase7_stats_p5_vs_baseline.csv', index=False
)
print('Saved: phase7_stats_p5_vs_baseline.csv')

print('\nAll CSVs saved (deterministic sorted)')

SAVING CSV ARTIFACTS
Saved: phase7_stats_ndcg_vs_baseline.csv
Saved: phase7_stats_failure_vs_baseline.csv
Saved: phase7_stats_p5_vs_baseline.csv

All CSVs saved (deterministic sorted)
